## 2 序列模型
### 2.1 理论计算题
给定字符序列：`ababc`，一阶马尔可夫 $p(x_t \mid x_{t-1})$，词汇表 $V=\{a,b,c\}$，**拉普拉斯加1平滑**。

拉普拉斯平滑条件概率公式：
$$
p(y \mid x) = \frac{\text{count}(x\to y)+1}{\sum_{z\in V}\big(\text{count}(x\to z)+1\big)}
= \frac{\text{count}(x\to y)+1}{\text{total\_trans\_from\_x}+|V|}
$$

#### 步骤1：统计原始转移计数
序列转移关系：$a\to b,\ b\to a,\ a\to b,\ b\to c$

转移统计表：
| 前驱$x$→后继$y$ | 次数 |
|----------------|------|
| $a\to b$       | 2    |
| $b\to a$       | 1    |
| $b\to c$       | 1    |

其余所有转移组合出现次数均为 $0$。词汇表大小 $|V|=3$。

#### 1. 计算 $p(a \mid b)$
- $\text{count}(b\to a)=1$
- 从 $b$ 出发原始总转移次数：
$$
\text{total\_trans\_from\_b} = \text{count}(b\to a)+\text{count}(b\to b)+\text{count}(b\to c) = 1+0+1=2
$$

代入平滑公式：
$$
p(a \mid b) = \frac{1+1}{2+3} = \frac{2}{5}
$$

#### 2. 计算 $p(c \mid b)$
- $\text{count}(b\to c)=1$
$$
p(c \mid b) = \frac{1+1}{2+3} = \frac{2}{5}
$$

**最终答案**
1. $p(a \mid b) = \boldsymbol{\dfrac{2}{5}}$
2. $p(c \mid b) = \boldsymbol{\dfrac{2}{5}}$

### 2.2 编程题：文本预处理函数 preprocess_text(text, n)

In [6]:
import string
from collections import Counter

def preprocess_text(text, n):
    # 1. 转小写，去除所有标点
    text_lower = text.lower()
    punc_table = str.maketrans("", "", string.punctuation)
    clean_text = text_lower.translate(punc_table)
    # 2. 空格分词
    tokens = clean_text.split()
    # 3. 按词频降序构建词汇表，ID从0开始
    word_counter = Counter(tokens)
    # 先按频次降序，频次相同按字母升序
    sorted_words = sorted(word_counter.keys(), key=lambda x: (-word_counter[x], x))
    vocab = {word: idx for idx, word in enumerate(sorted_words)}
    # 4. 滑动窗口生成n元特征与标签
    features = []
    labels = []
    max_start = len(tokens) - n
    for i in range(max_start):
        feat_win = tokens[i:i+n]
        label = tokens[i + n]
        features.append(feat_win)
        labels.append(label)
    return vocab, (features, labels)

# 测试用例
if __name__ == "__main__":
    test_input = "The time machine"
    vocab, (feat_list, label_list) = preprocess_text(test_input, n=2)
    print("词汇表 vocab：", vocab)
    print("特征列表 features：", feat_list)
    print("标签列表 labels：", label_list)

词汇表 vocab： {'machine': 0, 'the': 1, 'time': 2}
特征列表 features： [['the', 'time']]
标签列表 labels： ['machine']


## 3 循环神经网络
### 3.1 理论计算题：线性RNN 推导 $\dfrac{\partial L}{\partial W_{hh}}$
模型定义（无偏置）：
$$
h_t = W_{hh} h_{t-1} + W_{hx} x_t,\quad o_t = W_{oh} h_t
$$
损失：
$$
L = \frac12 \sum_{t=1}^T \|o_t - y_t\|_2^2
$$
令 $\delta_t = \dfrac{\partial L}{\partial h_t}$，由链式法则：
$$
\delta_t = \frac{\partial o_t}{\partial h_t}^\top \frac{\partial L}{\partial o_t} + W_{hh}^\top \delta_{t+1}
$$
其中 $\dfrac{\partial L}{\partial o_t}=o_t-y_t$，$\dfrac{\partial o_t}{\partial h_t}=W_{oh}$，故：
$$
\delta_t = W_{oh}^\top (o_t - y_t) + W_{hh}^\top \delta_{t+1},\quad \delta_{T+1}=0
$$

对 $W_{hh}$ 求梯度：
$$
\frac{\partial h_t}{\partial W_{hh}} = h_{t-1}^\top \otimes I
$$
$$
\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^T \delta_t h_{t-1}^\top
$$
完整展开（BPTT）：
$$
\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^T \left[ \left(\sum_{k=t}^T \big(W_{hh}^\top\big)^{k-t} W_{oh}^\top (o_k-y_k)\right) h_{t-1}^\top \right]
$$

#### 梯度消失/爆炸条件
设 $\lambda$ 为 $W_{hh}^\top$ 谱半径（最大特征值绝对值）：
1. 若 $\lambda < 1$：随时间步增长，$(W_{hh}^\top)^{k-t}$ 指数衰减 → **梯度消失**
2. 若 $\lambda > 1$：随时间步增长，$(W_{hh}^\top)^{k-t}$ 指数放大 → **梯度爆炸**


### 3.2 编程题：单步RNN前向+反向传播（tanh激活）

In [7]:
import numpy as np

def rnn_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    单步RNN前向传播 tanh激活
    x_t: (batch_size, input_size)
    h_prev: (batch_size, hidden_size)
    W_hx: (hidden_size, input_size)
    W_hh: (hidden_size, hidden_size)
    b_h: (hidden_size,)
    return h_t, pre_h
    """
    pre_h = x_t @ W_hx.T + h_prev @ W_hh.T + b_h
    h_t = np.tanh(pre_h)
    return h_t, pre_h

def rnn_backward(dh_next, x_t, h_prev, h_t, pre_h, W_hx, W_hh):
    """
    反向传播，上游梯度dL/dh_t = dh_next
    return dx_t, dh_prev, dW_hx, dW_hh, db_h
    """
    dtanh_grad = 1 - np.square(h_t)
    dpre_h = dh_next * dtanh_grad

    db_h = np.sum(dpre_h, axis=0)
    dW_hx = dpre_h.T @ x_t
    dW_hh = dpre_h.T @ h_prev
    dx_t = dpre_h @ W_hx
    dh_prev = dpre_h @ W_hh
    return dx_t, dh_prev, dW_hx, dW_hh, db_h

# 测试代码
if __name__ == "__main__":
    batch_size, input_size, hidden_size = 2, 3, 4
    # 随机初始化参数
    x_t = np.random.randn(batch_size, input_size)
    h_prev = np.random.randn(batch_size, hidden_size)
    W_hx = np.random.randn(hidden_size, input_size)
    W_hh = np.random.randn(hidden_size, hidden_size)
    b_h = np.random.randn(hidden_size)

    h_t, pre_h = rnn_forward(x_t, h_prev, W_hx, W_hh, b_h)
    # 模拟上游梯度
    dh_next = np.random.randn(batch_size, hidden_size)
    dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_backward(dh_next, x_t, h_prev, h_t, pre_h, W_hx, W_hh)
    print("h_t shape:", h_t.shape)
    print("dW_hh shape:", dW_hh.shape)

h_t shape: (2, 4)
dW_hh shape: (4, 4)


## 4 高级循环神经网络
### 4.1 理论计算题：深度双向RNN总参数量
参数设定：
- $L$：层数，$H$：单方向隐藏单元数，$D$：输入维度，$O$：输出维度
双向每层包含**前向RNN**+**后向RNN**；忽略嵌入层，仅计算RNN层+最终输出层。

单层单向RNN参数（权重+偏置）：
$$
\text{SingleDirParams} = (D\cdot H + H\cdot H) + H
$$
单层双向RNN：
$$
\text{BiLayerParams} = 2\cdot \big(DH + H^2 + H\big)
$$
$L$层双向堆叠：注意第2~L层输入维度为$2H$（拼接前向+后向）
- 第1层双向：$2(DH + H^2 + H)$
- 第$2\sim L$层双向：每层输入$2H$，$2\big(2H\cdot H + H^2 + H\big)$
- 最终输出层：输入$2H$，输出$O$：$2H\cdot O + O$

总参数表达式：
$$
\begin{aligned}
\text{TotalParams}
&= 2(DH+H^2+H) + (L-1)\cdot 2(2H^2+H^2+H) + (2HO+O) \\
&= 2DH + 2H^2 + 2H + 2(L-1)(3H^2+H) + O(2H+1)
\end{aligned}
$$

### 4.2 编程题：双向RNN编码器

In [8]:
import torch
import torch.nn as nn

class BiRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            bidirectional=True,
            batch_first=False
        )
        self.hidden_dim = hidden_dim

    def forward(self, X):
        """
        X: (seq_len, batch, input_dim)
        return seq_out, final_h
        seq_out: (seq_len, batch, 2*hidden_dim) 每一步拼接前向+后向隐状态
        final_h: (batch, 2*hidden_dim) 最后一步拼接全局序列表示
        """
        seq_output, h_n = self.rnn(X)
        # h_n shape: (2, batch, hidden_dim) 0前向，1后向
        forward_h = h_n[0]
        backward_h = h_n[1]
        final_h = torch.cat([forward_h, backward_h], dim=-1)
        return seq_output, final_h

# 测试
if __name__ == "__main__":
    seq_len, batch, input_dim, hidden_dim = 5, 3, 4, 2
    X = torch.randn(seq_len, batch, input_dim)
    encoder = BiRNNEncoder(input_dim, hidden_dim)
    seq_out, final_h = encoder(X)
    print("逐时间步拼接输出 shape:", seq_out.shape)
    print("全局序列表示 shape:", final_h.shape)

逐时间步拼接输出 shape: torch.Size([5, 3, 4])
全局序列表示 shape: torch.Size([3, 4])


## 5 嵌入向量
### 5.1 理论计算题：Skip-gram负采样损失
符号定义：
- $v_c$：中心词输入向量；$u_o$：正上下文输出向量；$u_{n_k}$：第$k$个负样本输出向量
- $K$：负样本数量；$\sigma(z)=\dfrac{1}{1+e^{-z}}$ sigmoid

单个$(w_c,w_o)$样本对数似然目标：
$$
\mathcal{L} = \log\sigma(u_o^\top v_c) + \sum_{k=1}^K \log\big(1-\sigma(u_{n_k}^\top v_c)\big)
$$
整体损失（最大化似然，训练最小化负对数似然）：
$$
\mathcal{J} = -\frac{1}{N}\sum_{(w_c,w_o)\in \mathcal{D}} \left[ \log\sigma(u_o^\top v_c) + \sum_{k=1}^K \log\big(1-\sigma(u_{n_k}^\top v_c)\big) \right]
$$

#### 负样本采样方式
从**噪声分布 $P_n(w)$** 采样，通常采用：
1. 一元分布（词频3/4次幂平滑）：$P_n(w) \propto \text{count}(w)^{3/4}$
2. 均匀随机采样（简单替代）；采样时排除正上下文词，避免重复。

### 5.2 编程题：CBOW 完整Softmax前向+交叉熵损失

In [9]:
import torch
import torch.nn.functional as F

def cbow_forward_loss(context_indices, target_idx, V, d, W, W_out):
    """
    context_indices: list[int] 单个样本上下文词索引
    target_idx: int 中心词索引
    V: 词汇表大小
    d: 嵌入维度
    W: (V, d) 输入嵌入矩阵
    W_out: (d, V) 输出权重矩阵
    return loss 标量损失
    """
    # 取出上下文向量并求均值
    context_embeds = W[context_indices]
    hidden_vec = torch.mean(context_embeds, dim=0)
    # 计算得分logits
    logits = hidden_vec @ W_out
    # 交叉熵损失
    loss = F.cross_entropy(logits.unsqueeze(0), torch.tensor([target_idx]))
    return loss

# 测试
if __name__ == "__main__":
    vocab_size, embed_dim = 10, 3
    W = torch.randn(vocab_size, embed_dim, requires_grad=True)
    W_out = torch.randn(embed_dim, vocab_size, requires_grad=True)
    ctx_words = [0, 2, 4]
    center_word = 1
    loss_val = cbow_forward_loss(ctx_words, center_word, vocab_size, embed_dim, W, W_out)
    print("CBOW交叉熵损失值：", loss_val.item())

CBOW交叉熵损失值： 3.244619607925415


## 6 注意力机制
### 6.1 理论计算题：缩放点积注意力
已知：
$Q\in\mathbb{R}^{2\times4},\ K\in\mathbb{R}^{3\times4},\ V\in\mathbb{R}^{3\times5},\ d_k=4$
$$
\text{Score} = \frac{QK^\top}{\sqrt{d_k}},\quad \text{AttnOutput} = \text{softmax}(\text{Score}) V
$$

#### 步骤1：计算 $QK^\top$
$QK^\top \in \mathbb{R}^{2\times3}$
$$
QK^\top =
\begin{bmatrix}
q_{11}k_{11}+q_{12}k_{12}+q_{13}k_{13}+q_{14}k_{14} & q_1^\top k_2 & q_1^\top k_3 \\
q_2^\top k_1 & q_2^\top k_2 & q_2^\top k_3
\end{bmatrix}
$$

#### 步骤2：缩放
$$
\text{Score} = \frac{QK^\top}{\sqrt{4}} = \frac{QK^\top}{2}
$$

#### 步骤3：逐行Softmax
对Score每行做softmax，得到权重矩阵 $\text{AttnWeight}\in\mathbb{R}^{2\times3}$，每行和为1：
$$
\text{AttnWeight}_{i,j} = \frac{\exp(\text{Score}_{i,j})}{\sum_{m=1}^3 \exp(\text{Score}_{i,m})}
$$

#### 步骤4：加权求和V
$$
\text{Output} = \text{AttnWeight} \cdot V \in \mathbb{R}^{2\times5}
$$

### 6.2 编程题：多头注意力 num_heads=2, d_model=4

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=4, num_heads=2):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        # QKV线性投影层
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        # 输出融合层
        self.w_o = nn.Linear(d_model, d_model)

    def scaled_dot_product_attn(self, Q, K, V):
        dk = torch.tensor(Q.size(-1), dtype=torch.float32)
        score = torch.matmul(Q, K.transpose(-2, -1)) / torch.sqrt(dk)
        attn_weight = F.softmax(score, dim=-1)
        attn_out = torch.matmul(attn_weight, V)
        return attn_out

    def forward(self, X):
        # X shape: (seq_len, batch, d_model)
        seq_len, batch, _ = X.shape
        # 投影QKV
        Q = self.w_q(X)
        K = self.w_k(X)
        V = self.w_v(X)
        # 分头：(seq_len, batch, heads, dk) -> (heads, batch, seq_len, dk)
        Q = Q.view(seq_len, batch, self.num_heads, self.d_k).permute(2, 1, 0, 3)
        K = K.view(seq_len, batch, self.num_heads, self.d_k).permute(2, 1, 0, 3)
        V = V.view(seq_len, batch, self.num_heads, self.d_k).permute(2, 1, 0, 3)
        # 每个头独立计算缩放点积注意力
        head_outputs = self.scaled_dot_product_attn(Q, K, V)
        # 还原维度，拼接多头
        head_outputs = head_outputs.permute(2, 1, 0, 3).contiguous()
        concat_out = head_outputs.view(seq_len, batch, self.d_model)
        # 最终线性变换
        final_out = self.w_o(concat_out)
        return final_out

# 测试
if __name__ == "__main__":
    seq_len, batch_size = 6, 2
    X = torch.randn(seq_len, batch_size, 4)
    mha_layer = MultiHeadAttention(d_model=4, num_heads=2)
    output = mha_layer(X)
    print("多头注意力输出 shape:", output.shape)

多头注意力输出 shape: torch.Size([6, 2, 4])
